In [ ]:
# ===================== Imports =====================
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd

from collections import defaultdict
from scipy.stats import norm, pearsonr

from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from pathlib import Path

from scipy.spatial.distance import pdist


sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from encoders import *
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.runners_v2 import (
    run_experiment_grid,
    run_experiment_scores,
    run_experiment_scores_itemwise,
    run_experiment_itemwise_hits_fas,
    make_noise_schedule
)

from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *


def load_config(cfg_path=None):
    """
    Load YAML config.
    Priority:
      1. cfg_path argument (if provided)
      2. sys.argv[1]
    """
    if cfg_path is None:
        if len(sys.argv) != 2:
            raise RuntimeError("Usage: python main.py path/to/run.yaml")
        cfg_path = sys.argv[1]

    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    with open(cfg_path, "r") as f:
        return yaml.safe_load(f), cfg_path


tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

In [ ]:
model_cfg, model_cfg_path = load_config("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/model_yamls/all_nn_v4/run_000002.yaml")

noise_cfg = model_cfg["noise_model"]
print(model_cfg['experiment']['n_seqs'])
print(model_cfg)

assert "t_step" in noise_cfg, "t_step is needed for two-regime"

In [ ]:
factor = 75
epsilon = 0.0005

model_cfg['noise_model']['sigma0_max'] = 0.295
model_cfg['noise_model']['sigma1_max'] = 0.022 + epsilon

model_cfg['noise_model']['sigma0_min'] = 0.200
model_cfg['noise_model']['sigma1_min'] = 0.019 - epsilon
model_cfg['noise_model']['t_step'] = 3

print(model_cfg['noise_model'])

In [ ]:
xs_log = []
xs = []
for _ in range(90000):
    x_log = np.random.uniform(np.log(0.001), np.log(1))
    xs_log.append(np.exp(x_log))

    x = np.random.uniform(0.001, 1)
    xs.append(x)
plt.hist(xs_log, bins=100, alpha=0.4);
plt.hist(xs, bins=100, alpha=0.4);

In [ ]:
# ---------------------------
# SETTINGS (from YAML)
# ---------------------------
# ---- experiment ----
exp_cfg = model_cfg["experiment"]

which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)

if is_multi:
    isis = [0, 1, 2, 4, 8, 16, 32, 64]
else:
    assert which_isi is not None, "which_isi required if not multi-ISI"
    isis = [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
print(noise_cfg)

# ---- representation ----
repr_cfg = model_cfg["representation"]

encoder_type = repr_cfg["type"]
layer   = repr_cfg.get("layer", None)
PC_DIMS = repr_cfg.get("pc_dims", None)

# ---------------------------
# 1. Load data
# ---------------------------
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(
    which_task, which_isi, is_multi, old=False)

# ---------------------------
# 2. Human curve
# ---------------------------
human_curve = compute_human_curve(human_runs, is_multi, which_isi)
results_root = model_cfg["results_root"]
tag = model_cfg.get("tag", "untagged")

if noise_mode == "two-regime":
    assert "t_step" in noise_cfg, "two-regime requires t_step"
    t_step = noise_cfg["t_step"]
    noise_tag = f"{noise_mode}_t{t_step}"
else:
    noise_tag = noise_mode

save_figs = (
    f"{results_root}/"
    f"figures/test_v3/"
    f"{task_name}/"
    f"{encoder_type}/"
    f"{metric}/"
    f"{noise_tag}/"
)

save_fits = f"{results_root}/fits/test_v3"


print(save_figs, save_fits)
ensure_dir(save_figs)

ensure_dir(save_fits)


In [ ]:
def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    """
    Estimate the median pairwise distance under a given metric.

    Args:
        X (np.ndarray): shape (N, D). Must already be preprocessed
                        appropriately for the metric
                        (e.g., L2-normalized for cosine).
        metric (str): distance metric for scipy.spatial.distance.pdist
                      ("cosine", "euclidean", "mahalanobis", etc.).
        n_samples (int): number of items to subsample for efficiency.
        seed (int): RNG seed.

    Returns:
        float: median pairwise distance.
    """
    rng = np.random.default_rng(seed)
    N = X.shape[0]

    idx = rng.choice(N, size=min(n_samples, N), replace=False)
    Xs = X[idx]

    dists = pdist(Xs, metric=metric)
    d50 = np.median(dists)

    return float(d50)


save_figs = (
    f"{results_root}/"
    f"figures/test/v5/"
    f"{task_name}/"
    f"{encoder_type}/"
    f"{metric}/"
    f"{noise_tag}/"
)

save_fits = f"{results_root}/fits/test/v5_best"
save_fits_all = f"{results_root}/fits/test/v5_all"

ensure_dir(save_figs)
ensure_dir(save_fits_all)
ensure_dir(save_fits)


encoder_type = repr_cfg["type"]     # e.g. "resnet50"
layer        = repr_cfg.get("layer")
pc_dims      = repr_cfg.get("pc_dims")

print(layer)

NN_ENCODERS = {"kell2018", "resnet50"}

encoder_task = (
    "word_speaker_audioset"
    if encoder_type in NN_ENCODERS
    else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,      # e.g. "resnet50"
    model_name=encoder_type,        # same by design
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    device="cuda",
)

# ---- representation-specific fields ----
if encoder_type in ("kell2018", "resnet50"):
    encoder_cfg["layer"] = layer
    assert layer is not None, f"layer required for {encoder_type}"

if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims
    assert pc_dims is not None, "pc_dims required for texture encoder"

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)

# X is torch tensor at this point
X_np = X.detach().cpu().numpy()

d50 = median_pairwise_distance(
    X_np,
    metric="cosine",      # e.g. "cosine", "euclidean", "mahalanobis"
    n_samples=500,
    seed=0,
)

d50 = 1

print(d50)
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]

# ---------------------------
# Noise parameter bounds (from YAML)
# ---------------------------

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"]*d50, noise_cfg["sigma0_max"]*d50),
}

if noise_mode == "two-regime":
    param_bounds["sigma1"] = (
        noise_cfg["sigma1_min"]*d50,
        noise_cfg["sigma1_max"]*d50,
    )
    param_bounds["t_step"] = (
        noise_cfg["t_step"],
        noise_cfg["t_step"],
    )

import time
start_time = time.time()
opt_results = random_search_gridlike(
    n_samples=5,
    param_bounds=param_bounds,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list,
    isis=isis,
    human_curve=human_curve,
    layer=encoder_name,          # legacy arg, keep for now
    encoder_name=encoder_name,   # canonical identifier
    hr_task_name=hr_task_name,
    debug=False,
)
print("--- %s seconds ---" % (time.time() - start_time))

best_fits = generate_and_plot_model_decay_summary_v2(
    opt_results,
    human_curve,
    isis,
    savedir=save_figs,
    max_rows=1,
    verbose=True, hr_task_name=hr_task_name, encoder_name=encoder_name)

summary_list = save_best_models(best_fits, save_fits_all, prefix=f"{task_name}-{encoder_name}")

#plot_best_model_histograms(best_fits, isis, save_figs)


In [ ]:
for opt_result in opt_results:
    print(opt_result['params']['sigma0'], opt_result['nmse'])

In [ ]:
def l2_distance(a, b):
    """Euclidean (L2) distance between two vectors."""
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.linalg.norm(a - b))

def l1_distance(a, b):
    """L1 (Manhattan) distance between two vectors."""
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.sum(np.abs(a - b)))

from scipy.stats import pearsonr

def pearson_r(a, b):
    """Pearson correlation between two vectors."""
    a = np.asarray(a)
    b = np.asarray(b)

    if np.all(np.isclose(a, a[0])) or np.all(np.isclose(b, b[0])):
        return np.nan  # undefined if flat

    r, _ = pearsonr(a, b)
    return float(r)

def slope_on_log_isi(dp, isis):
    """
    Compute finite-difference slope of d' vs log(ISI),
    excluding ISI = 0.
    """
    dp = np.asarray(dp)
    isis = np.asarray(isis)

    valid = isis > 0
    dp = dp[valid]
    isis = isis[valid]

    logt = np.log(isis)
    return np.diff(dp) / np.diff(logt)
    
def slope_l2_distance(a, b, isis):
    """
    L2 distance between slopes of two d' curves on log(ISI),
    excluding ISI = 0.
    """
    sa = slope_on_log_isi(a, isis)
    sb = slope_on_log_isi(b, isis)

    if len(sa) == 0 or len(sb) == 0:
        return np.nan

    return float(np.linalg.norm(sa - sb))

for rec in opt_results:
    model_dp = rec.get("model_dp", None)

    if model_dp is None:
        rec["l2_dp"] = np.nan
        rec["l1_dp"] = np.nan
        continue

    rec["l2_dp"] = l2_distance(model_dp, human_curve)
    rec["l1_dp"] = l1_distance(model_dp, human_curve)

for rec in opt_results:
    model_dp = rec.get("model_dp", None)

    if model_dp is None:
        rec["pearson_r"] = np.nan
        rec["slope_l2"] = np.nan
        continue

    rec["pearson_r"] = pearson_r(model_dp, human_curve)
    rec["slope_l2"] = slope_l2_distance(model_dp, human_curve, isis)

dp_scale = np.linalg.norm(human_curve) + 1e-12

for rec in opt_results:
    model_dp = rec.get("model_dp", None)
    if model_dp is None:
        continue

    rec["l2_dp_norm"] = rec["l2_dp"] / dp_scale
    rec["l1_dp_norm"] = rec["l1_dp"] / (np.sum(np.abs(human_curve)) + 1e-12)

In [ ]:
import numpy as np

def metric_to_alpha(values, higher_is_better, alpha_min=0.05, alpha_max=0.9):
    """
    Map metric values to alpha transparencies.
    Best model -> alpha_max, worst -> alpha_min.
    """
    v = np.asarray(values, float)

    if higher_is_better:
        scores = v
    else:
        scores = -v  # invert so higher is better

    lo, hi = np.nanmin(scores), np.nanmax(scores)
    if np.isclose(lo, hi):
        return np.full_like(scores, alpha_max)

    a = (scores - lo) / (hi - lo)
    return alpha_min + a * (alpha_max - alpha_min)

import matplotlib.pyplot as plt

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def plot_model_comparisons_by_metric(
    opt_results,
    human_dp,
    isis,
    metric_keys,
    model_dp_key="model_dp",
    title_prefix="Model vs Human",
    alpha_min=0.03,
    alpha_max=0.95,
    alpha_gamma=2.5,
    cmap="viridis",
):
    """
    Plot model d' curves with color + alpha encoding per metric.
    Layout: 2 columns, multiple rows.
    Each model plotted with line + dots at ISIs.
    Color and alpha both reflect metric goodness.
    """

    def metric_to_visuals(values, higher_is_better):
        v = np.asarray(values, float)
        scores = v if higher_is_better else -v
        lo, hi = np.nanmin(scores), np.nanmax(scores)

        if np.isclose(lo, hi):
            normed = np.ones_like(scores)
        else:
            normed = (scores - lo) / (hi - lo)
            normed = np.clip(normed, 0, 1)

        # alpha: nonlinear emphasis of best models
        alpha = alpha_min + (normed ** alpha_gamma) * (alpha_max - alpha_min)

        return normed, alpha, Normalize(vmin=lo, vmax=hi)

    n_metrics = len(metric_keys)
    n_cols = 2
    n_rows = math.ceil(n_metrics / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharey=False
    )

    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)

    for idx, key in enumerate(metric_keys):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]

        metric_vals = []
        curves = []
        
        for rec in opt_results:
            if key not in rec or model_dp_key not in rec:
                continue
        
            val = rec[key]
            if not np.isfinite(val):
                continue   # <-- CRITICAL FIX
        
            metric_vals.append(val)
            curves.append(np.asarray(rec[model_dp_key]))

        metric_vals = np.asarray(metric_vals)

        higher_is_better = (
            "cos" in key.lower() or
            "sim" in key.lower()
        )

        normed, alphas, norm = metric_to_visuals(
            metric_vals, higher_is_better
        )

        cmap_obj = plt.get_cmap(cmap)

        all_y = []

        # ---- plot models ----
        for dp, g, a in zip(curves, normed, alphas):
            color = cmap_obj(g)
            ax.plot(
                isis, dp,
                color=color,
                alpha=a,
                lw=1.2
            )
            ax.scatter(
                isis, dp,
                color=color,
                alpha=a,
                s=18
            )
            all_y.append(dp)

        # ---- plot human ----
        ax.plot(
            isis, human_dp,
            color="red",
            lw=2.8,
            zorder=10,
            label="Human",
            alpha=0.3
        )
        ax.scatter(
            isis, human_dp,
            color="black",
            s=40,
            zorder=11,
            alpha=0.3
        )
        all_y.append(human_dp)

        # ---- robust y-limits ----
        all_y = np.concatenate(all_y)
        ylo, yhi = np.nanpercentile(all_y, [1, 99])
        pad = 0.1 * (yhi - ylo + 1e-6)
        ax.set_ylim(ylo - pad, yhi + pad)

        ax.set_title(f"{title_prefix}\nmetric = {key}")
        ax.set_xlabel("ISI")
        ax.grid(alpha=0.2)

        # ---- colorbar (now directly maps to curve color) ----
        sm = ScalarMappable(norm=norm, cmap=cmap_obj)
        sm.set_array([])

        cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(
            "Better ⟵ metric ⟶ Worse" if not higher_is_better else
            "Worse ⟵ metric ⟶ Better",
            rotation=270,
            labelpad=12
        )

    # turn off unused axes
    for j in range(idx + 1, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        axes[r, c].axis("off")

    axes[0, 0].set_ylabel("Sensitivity (d')")
    axes[0, 0].legend(frameon=False)

    plt.tight_layout()
    plt.show()

def extract_sigma_metric_table(opt_results, metric_key):
    """
    Return a DataFrame with columns:
    ['sigma0', 'sigma1', metric_key]
    """
    rows = []

    for rec in opt_results:
        params = rec.get("params", {})
        if metric_key not in rec:
            continue

        sigma0 = params.get("sigma0", np.nan)
        sigma1 = params.get("sigma1", np.nan)

        rows.append({
            "sigma0": sigma0,
            "sigma1": sigma1,
            metric_key: rec[metric_key],
        })

    return pd.DataFrame(rows)

metric_keys = [
    "nmse",
    "l2_dp",
    "l1_dp",
    "pearson_r",
    "slope_l2", 
    "cosine_sim"
]

plot_model_comparisons_by_metric(
    opt_results=opt_results,
    human_dp=human_curve,
    isis=isis,
    metric_keys=metric_keys,
)

In [ ]:
from matplotlib.ticker import ScalarFormatter, LogLocator

def plot_metric_landscapes_over_sigmas(
    opt_results,
    metric_keys,
    title_prefix="Metric landscape",
    cmap="cividis",
):
    """
    Plot metric landscapes over (sigma0, sigma1).
    Axes are log-scaled, but tick labels show raw sigma values.
    One subplot per metric (2 columns).
    """

    n_metrics = len(metric_keys)
    n_cols = 2
    n_rows = math.ceil(n_metrics / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=True,
        sharey=True,
    )

    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)

    for idx, key in enumerate(metric_keys):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]

        sig0, sig1, vals = [], [], []

        for rec in opt_results:
            params = rec.get("params", {})
            if key not in rec or "sigma0" not in params:
                continue

            sig0.append(params["sigma0"])
            sig1.append(params.get("sigma1", np.nan))
            vals.append(rec[key])

        sig0 = np.asarray(sig0)
        sig1 = np.asarray(sig1)
        vals = np.asarray(vals)

        sc = ax.scatter(
            sig0,
            sig1,
            c=vals,
            cmap=cmap,
            s=45,
            alpha=0.85,
        )

        # ---- log scaling ----
        # ax.set_xscale("log")
        # ax.set_yscale("log")

        # ---- major ticks: powers of 10 ----
        ax.xaxis.set_major_locator(LogLocator(base=10.0))
        ax.yaxis.set_major_locator(LogLocator(base=10.0))

        # ---- minor ticks: 2, 3, 5 within each decade ----
        ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=(2, 3, 5)))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=(2, 3, 5)))

        # ---- readable labels (no scientific notation) ----
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.ticklabel_format(style="plain", axis="both")

        ax.set_title(key)
        ax.grid(alpha=0.25, which="both")

        cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(key, rotation=270, labelpad=12)

    # turn off unused axes
    for j in range(idx + 1, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        axes[r, c].axis("off")

    axes[-1, 0].set_xlabel("sigma0")
    axes[-1, 1].set_xlabel("sigma0")
    axes[0, 0].set_ylabel("sigma1")

    plt.suptitle(title_prefix, y=1.02)
    plt.tight_layout()
    plt.show()

def plot_metric_vs_sigma_ratio(
    opt_results,
    metric_keys,
    title_prefix="Metric vs sigma0/sigma1",
    cmap="tab10",
):
    """
    Plot metric value vs sigma0/sigma1 ratio.
    One subplot per metric (2 columns).
    """

    n_metrics = len(metric_keys)
    n_cols = 2
    n_rows = math.ceil(n_metrics / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=True,
        sharey=False,
    )

    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)

    for idx, key in enumerate(metric_keys):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]

        ratios = []
        vals = []

        for rec in opt_results:
            params = rec.get("params", {})
            if key not in rec:
                continue

            s0 = params.get("sigma0", None)
            s1 = params.get("sigma1", None)
            if s0 is None or s1 is None or s1 <= 0:
                continue

            ratios.append(s0 / s1)
            vals.append(rec[key])

        ratios = np.asarray(ratios)
        vals = np.asarray(vals)

        ax.scatter(
            ratios,
            vals,
            s=45,
            alpha=0.8,
        )

        # log-scale ratio, readable labels
        ax.set_xscale("log")
        ax.xaxis.set_major_locator(LogLocator(base=10.0))
        ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=(2, 3, 5)))
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.ticklabel_format(style="plain", axis="x")

        ax.set_title(key)
        ax.set_xlabel("sigma0 / sigma1")
        ax.set_ylabel(key)
        ax.grid(alpha=0.25, which="both")

    # turn off unused axes
    for j in range(idx + 1, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        axes[r, c].axis("off")

    plt.suptitle(title_prefix, y=1.02)
    plt.tight_layout()
    plt.show()

metric_keys = [
    "nmse",
    "l2_dp",
    "l1_dp",
    "pearson_r",
    "slope_l2", 
    "cosine_sim"
]

plot_metric_landscapes_over_sigmas(
    opt_results,
    metric_keys,
    title_prefix="Model quality over (σ₀, σ₁)",
)

plot_metric_vs_sigma_ratio(
    opt_results,
    metric_keys,
    title_prefix="Model quality vs σ₀ / σ₁",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _prep_isi_curve(isis, dp, exclude_zero=True, use_log_time=True):
    """Prepare x (time axis) and y (d') for knee/inflection analysis."""
    isis = np.asarray(isis, float)
    dp = np.asarray(dp, float)

    keep = np.isfinite(isis) & np.isfinite(dp)
    if exclude_zero:
        keep &= isis > 0

    isis_k = isis[keep]
    dp_k = dp[keep]

    if use_log_time:
        x = np.log(isis_k)
    else:
        x = isis_k

    return x, dp_k, isis_k, keep


def fit_piecewise_linear_knee(isis, dp, *, min_points_per_side=2, exclude_zero=True, use_log_time=True):
    """
    Find the best 2-segment piecewise linear fit (continuous at breakpoint)
    to d'(ISI), typically on log(ISI).

    Returns:
        dict with:
            'knee_isi', 'knee_x', 'slope1', 'slope2', 'intercept1', 'intercept2',
            'sse', 'yhat', 'keep_mask'
    """
    x, y, isis_k, keep_mask = _prep_isi_curve(isis, dp, exclude_zero=exclude_zero, use_log_time=use_log_time)
    n = len(x)
    if n < 2 * min_points_per_side + 1:
        raise ValueError(f"Not enough points for 2-segment fit. Need at least {2*min_points_per_side+1}, got {n}.")

    # Candidate breakpoints are interior indices (not endpoints)
    candidate_js = range(min_points_per_side, n - min_points_per_side)

    best = None
    for j in candidate_js:
        xb = x[j]  # breakpoint location in x-space

        # Build a continuous piecewise linear basis:
        # y ≈ a + b*x + c*max(0, x - xb)
        # where slope left = b, slope right = b + c
        hinge = np.maximum(0.0, x - xb)
        A = np.column_stack([np.ones_like(x), x, hinge])

        # Least squares fit
        coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        a, b, c = coef

        yhat = A @ coef
        sse = float(np.sum((y - yhat) ** 2))

        if (best is None) or (sse < best["sse"]):
            best = dict(
                knee_x=float(xb),
                knee_isi=float(np.exp(xb)) if use_log_time else float(xb),
                slope1=float(b),
                slope2=float(b + c),
                intercept1=float(a),
                # For the right segment, intercept depends on xb (enforcing continuity)
                intercept2=float(a - c * xb),
                sse=sse,
                yhat=yhat.copy(),
                keep_mask=keep_mask.copy(),
                x=x.copy(),
                y=y.copy(),
                isis_k=isis_k.copy(),
            )

    return best


def plot_knee_fit(isis, dp, fit, *, title="Human d' vs ISI with knee fit", show_original_zero=True):
    """Plot the data and the best piecewise-linear fit (on x used in fitting)."""
    isis = np.asarray(isis, float)
    dp = np.asarray(dp, float)

    x = fit["x"]
    y = fit["y"]
    isis_k = fit["isis_k"]

    # Reconstruct fitted curve in x-space, then plot against ISI (original scale)
    yhat = fit["yhat"]
    knee_isi = fit["knee_isi"]

    plt.figure(figsize=(6, 4))
    plt.plot(isis_k, y, marker="o", linestyle="None", label="Human (used)")
    plt.plot(isis_k, yhat, marker="o", linestyle="-", label="Piecewise fit")

    if show_original_zero and np.any(isis == 0):
        plt.plot(isis[isis == 0], dp[isis == 0], marker="o", linestyle="None", label="ISI=0 (excluded)")

    plt.axvline(knee_isi, linestyle="--", label=f"knee @ {knee_isi:.3g}")
    plt.xscale("log")  # show ISI naturally (no log numbers if you prefer linear ticks)
    plt.xlabel("ISI")
    plt.ylabel("d'")
    plt.title(title + f"\nslopes: early={fit['slope1']:.3g}, late={fit['slope2']:.3g}")
    plt.grid(alpha=0.25, which="both")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def bootstrap_knee(isis, dp, *, n_boot=2000, seed=0, **fit_kwargs):
    """
    Bootstrap the knee by resampling points with replacement.
    Note: with few ISIs this is a rough uncertainty estimate (still useful).
    """
    rng = np.random.default_rng(seed)
    isis = np.asarray(isis, float)
    dp = np.asarray(dp, float)

    # Use the same keep mask logic as the fitter
    x, y, isis_k, keep_mask = _prep_isi_curve(
        isis, dp,
        exclude_zero=fit_kwargs.get("exclude_zero", True),
        use_log_time=fit_kwargs.get("use_log_time", True),
    )

    n = len(isis_k)
    knees = []
    slopes1 = []
    slopes2 = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        fit = fit_piecewise_linear_knee(isis_k[idx], y[idx], exclude_zero=False, use_log_time=fit_kwargs.get("use_log_time", True),
                                        min_points_per_side=fit_kwargs.get("min_points_per_side", 2))
        knees.append(fit["knee_isi"])
        slopes1.append(fit["slope1"])
        slopes2.append(fit["slope2"])

    knees = np.asarray(knees)
    slopes1 = np.asarray(slopes1)
    slopes2 = np.asarray(slopes2)

    def ci(a):
        return float(np.nanpercentile(a, 2.5)), float(np.nanpercentile(a, 97.5))

    return dict(
        knee_ci=ci(knees),
        slope1_ci=ci(slopes1),
        slope2_ci=ci(slopes2),
        knees=knees,
        slopes1=slopes1,
        slopes2=slopes2,
    )


# --------------------
# Example usage
# --------------------


fit = fit_piecewise_linear_knee(isis, human_curve, min_points_per_side=2, exclude_zero=True, use_log_time=True)
print("knee ISI:", fit["knee_isi"])
print("early slope (d' per log time):", fit["slope1"])
print("late slope  (d' per log time):", fit["slope2"])
plot_knee_fit(isis, human_curve, fit)

boot = bootstrap_knee(isis, human_curve, n_boot=1000, seed=0, min_points_per_side=1, exclude_zero=True, use_log_time=False)
print("knee 95% CI:", boot["knee_ci"])

In [ ]:
def refine_search_local(
    base_results,
    n_refine,
    param_bounds,
    *,
    noise_mode,
    top_k=10,
    jitter_frac=0.15,
    rng=None
):
    rng = np.random.default_rng(rng)

    # Sort by coarse fit (lower is better)
    base_results = sorted(base_results, key=lambda r: r["nmse"])

    # Keep only top-K elites
    elites = base_results[:top_k]

    refined_params = []

    for base in elites:
        p0 = base["params"]
        print("p0", p0)

        for _ in range(n_refine):
            p = {}
            for k, (lo, hi) in param_bounds.items():
                span = hi - lo
                val = p0[k] + rng.normal(scale=jitter_frac * span)
                p[k] = np.clip(val, lo, hi)

            if noise_mode == "two-regime":
                if p["sigma0"] <= p["sigma1"]:
                    continue

            refined_params.append(p)

    return refined_params

In [ ]:
refined_params = refine_search_local(
    base_results=opt_results,
    top_k=3,
    n_refine=20,
    param_bounds=param_bounds,
    noise_mode=noise_mode,
)



In [ ]:
#list(refined_params)

In [ ]:
def evaluate_params(
    params_list,
    *,
    noise_mode,
    metric,
    X0,
    name_to_idx,
    experiment_list,
    isis,
    human_curve,
    layer,
    encoder_name,
    hr_task_name,
    debug=False
):
    results = []

    for params in params_list:
        params = dict(params)  # defensive copy

        params.update({
            "noise_mode": noise_mode,
            "metric": metric,
            "layer": layer,
            "encoder": encoder_name,
            "stimulus_set": hr_task_name,
        })

        run_out = run_experiment_scores(
            sigma0=params["sigma0"],
            sigma1=params.get("sigma1", None),
            t_step=params.get("t_step", None),
            rate=params.get("rate", None),
            noise_mode=noise_mode,
            metric=metric,
            X0=X0,
            name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            debug=debug,
        )


        model_dp = compute_model_dprime_for_run(run_out, isis)

        results.append({
            "params": params,
            "results": run_out,
            "model_dp": model_dp,
            "nmse": compute_nmse(model_dp, human_curve),
            "nmse_no_0": compute_nmse(model_dp, human_curve, start_idx=1),
            "mse": compute_mse(model_dp, human_curve),
            "mse_no_0": compute_mse(model_dp, human_curve, start_idx=1),
        })

    return results

In [ ]:
# 3) Run full experiments on refined params
fine_results = evaluate_params(
    refined_params,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list[16:25],
    isis=isis,
    human_curve=human_curve,
    layer=layer,
    encoder_name=encoder_name,
    hr_task_name=hr_task_name,
)

summary_list = save_best_models(best_fits, save_fits, prefix=f"{task_name}-{encoder_name}")


In [ ]:
best_fits = generate_and_plot_model_decay_summary_v2(
    fine_results,
    human_curve,
    isis,
    savedir=save_figs,
    max_rows=1,
    verbose=True, hr_task_name=hr_task_name, encoder_name=encoder_name)

summary_list = save_best_models(best_fits, save_fits_all, prefix=f"{task_name}-{encoder_name}")

plot_best_model_histograms(best_fits, isis, save_figs)